# MINI Cells — Experiment 007: MiniCells-30M v0

Trains a 29.6M-parameter MiniCells model and a 29.46M parameter-matched Transformer through a 100M-token schedule. With two T4 GPUs the models run concurrently. Full training state is checkpointed every 5M tokens for exact resume.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')
if not (ROOT / '.git').exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(['git','clone','--depth','1','https://github.com/ArcheLabs/mini-cells.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)
    subprocess.run(['git','switch','main'], cwd=ROOT, check=True)
    subprocess.run(['git','reset','--hard','origin/main'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[lm]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git','rev-parse','--short','HEAD'], check=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))
if not torch.cuda.is_available(): raise RuntimeError('Experiment 007 requires CUDA')


In [ ]:
subprocess.run([sys.executable,'-m','pytest',
    'tests/research/01-foundations/test_language_bridge.py','tests/research/01-foundations/test_language_ablation.py',
    'tests/research/01-foundations/test_language_scaling.py','tests/research/01-foundations/test_language_30m.py','-q'], cwd=ROOT, check=True)


## Run / resume

`STOP_AFTER_TOKENS` defaults to the full 100M target. For a staged Kaggle run, set it to e.g. `25_000_000` or `50_000_000`. The LR schedule still targets 100M, so a later resume is exact rather than a warm restart.

If resuming from a prior Kaggle saved output/dataset, set `RESUME_INPUT` to the mounted directory containing the two `*-latest.pt` files (or its parent `resume` layout).


In [ ]:
STOP_AFTER_TOKENS = 100_000_000
RESUME_INPUT = ''  # example: '/kaggle/input/minicells-30m-resume/resume'
cmd = [sys.executable, 'scripts/research/run_consumer_language_30m.py', '--stop-after-tokens', str(STOP_AFTER_TOKENS)]
if RESUME_INPUT:
    cmd += ['--resume-input', RESUME_INPUT]
subprocess.run(cmd, cwd=ROOT, check=True)


In [ ]:
import json
from IPython.display import Image, Markdown, display
OUT = ROOT / 'results' / 'consumer-language-30m-v1'
progress = json.loads((OUT / 'progress.json').read_text(encoding='utf-8'))
display(Markdown(f"## Progress: {progress['consumed_tokens']} / {progress['target_tokens']}"))
if progress['complete']:
    decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
    display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
    display(Markdown(f"**100M PPL ratio:** {decision['comparison']['ppl_ratio_100m']:.4f}×  \
**Slope ratio:** {decision['comparison']['slope_ratio_to_transformer']:.4f}"))
    for name in ['ppl-scaling.png','nll-scaling.png','relative-gap.png','throughput.png']:
        display(Image(filename=str(OUT / name)))
    display(Markdown((OUT / 'generation-progression.md').read_text(encoding='utf-8')))
    display(Markdown((OUT / 'MODEL_CARD.md').read_text(encoding='utf-8')))
else:
    export_dir = Path('/kaggle/working/minicells-30m-resume-export')
    export_dir.mkdir(parents=True, exist_ok=True)
    for model in ['minicells-30m-v0','transformer-30m']:
        shutil.copy2(OUT / 'resume' / f'{model}-latest.pt', export_dir / f'{model}-latest.pt')
    print('Partial run. Preserve this directory with Kaggle Save Version/output:', export_dir)


In [ ]:
# Publish only after the run reaches 100M and you have reviewed the outputs.
PUBLISH = False
if PUBLISH:
    progress = json.loads((OUT / 'progress.json').read_text(encoding='utf-8'))
    if not progress['complete']: raise RuntimeError('Do not publish an incomplete Experiment 007 run')
    subprocess.run([sys.executable,'scripts/research/publish_experiment_007_results.py','--push'], cwd=ROOT, check=True)
